In [43]:
import sys
import os
if not '../..' in sys.path:
    sys.path.append('../..')
import basico
import pandas
import torch
import torch.nn as nn

In [ ]:
fname = "../ParkRam_constant_feed.xlsx"
data = pandas.read_excel(fname, sheet_name=1)

In [45]:
def convert_symbol_to_label(symbol):
    return f"{symbol}_noise"

def convert_symbol_to_sd(symbol):
    return f"sd_{symbol}"

In [46]:
with open("../PR_testbatch.txt", "r") as f:
    testbatch_str = f.read().split()
    testbatch = set([int(el) for el in testbatch_str])

In [47]:
prev_val = 0
batches_lengths = []
for val in data["exp"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1

not_testbatch = set(range(len(batches_lengths))) - testbatch
not_testbatch_exp_labels = [el + 1 for el in not_testbatch]

In [48]:
maxes = dict()
mins = dict()
concentrations_symbols = ["X", "S", "PT", "PM", "V"]
true_symbols = ["X True", "S true", "PT true", "PM true", "V true"]
concentrations_labels = [convert_symbol_to_label(symbol) for symbol in concentrations_symbols]

for label in (['t'] + concentrations_labels):
    max_val = max(data[data["exp"].isin(not_testbatch_exp_labels)][label])
    min_val = min(data[data["exp"].isin(not_testbatch_exp_labels)][label])
    label_scaled = f"{label}_scaled"
    label_scaled = f"{label}_scaled"
    maxes[label] = max_val
    mins[label] = min_val
    data[label_scaled] = (data[label] - min_val) / (max_val - min_val)

mins_tensor = torch.Tensor([mins[label] for label in concentrations_labels])
maxes_tensor = torch.Tensor([maxes[label] for label in concentrations_labels])

In [49]:
batches = []
for i, length in enumerate(batches_lengths):
    batches.append(data.iloc[i*length : i*length + length].reset_index(drop=True))
length = batches_lengths[0]

#scale inputs
changing_vars = ["X", "S", "V", "Feed"]

def make_label_from_changing_var(v):
    if v == "Feed":
        return v
    else:
        return f"{v}_noise"

changing_vars_labels = [make_label_from_changing_var(changing_var) for changing_var in changing_vars]
first_of_each_batch = data[changing_vars_labels].iloc[::length]
mins_Xs = first_of_each_batch.min()
maxes_Xs = first_of_each_batch.max()

def make_constant_input_label(label):
    return f"{label}0"

mins_Xs_dict = {make_constant_input_label(label): minX for label, minX in zip(changing_vars, mins_Xs)}
maxes_Xs_dict = {make_constant_input_label(label): maxX for label, maxX in zip(changing_vars, maxes_Xs)}
scaled_initial_conditions = (first_of_each_batch - mins_Xs)/(maxes_Xs-mins_Xs)

Xs_ys = []
for i, batch in enumerate(batches):
    scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
    X = pandas.concat([batch[["t_scaled"]], scaled_initials_for_this_batch], axis=1)
    y = batch[[f"{label}_scaled" for label in concentrations_labels]]
    Xs_ys.append((X, y))

In [50]:
def wmse(input, target, sd):
    return torch.sum(((input - target) / sd) ** 2) / input.numel()

loss_fn = wmse

In [51]:
X_all = torch.stack([torch.Tensor(batch[0].values) for batch in Xs_ys])

In [52]:
INPUTS = 5
OUTPUTS = 5

class PRRegressor_ffnn(nn.Module):
    def __init__(self, inputs, layers, outputs, activation):
        super().__init__()
        all_layers = [inputs] + layers + [outputs]
        self.linears = nn.ModuleList([nn.Linear(all_layers[i], all_layers[i + 1]) for i in range(len(all_layers) - 1)])
        if activation == "tanh":
            self.act = nn.Tanh()
        elif activation == "relu":
            self.act = nn.ReLU()
    
    def forward(self, x):
        for hidden in self.linears[:-1]:
            x = self.act(hidden(x))
        x = self.linears[-1](x)
        return x

nums_nodes_layers_s_ffnn = [[13, 13], [15, 15], [13, 13, 13]]

In [53]:
NUM_HIDDEN = 1

class PR_rnn(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, activation):
        super(PR_rnn, self).__init__()
        
        # Number of hidden dimensions
        self.hidden_dim = hidden_dim
        
        # Number of hidden layers
        self.layer_dim = layer_dim
        
        # RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, nonlinearity=activation)
        
        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim)
            
        # One time step
        out, hn = self.rnn(x, h0)
        out = self.fc(out) 
        return out

nums_nodes_one_layer_rnn = [10, 15, 19]

In [54]:
num_nodes = {
    "ffnn": nums_nodes_layers_s_ffnn,
    "rnn": nums_nodes_one_layer_rnn
}
activations = ["tanh", "relu"]

In [55]:
def load_model(model_name, path_to_folder, model_type, activation, layers):
    if model_type == "ffnn":
        nn_model = PRRegressor_ffnn(INPUTS, layers, OUTPUTS, activation)
    elif model_type == "rnn":
        nn_model = PR_rnn(INPUTS, layers, NUM_HIDDEN, OUTPUTS, activation)
    nn_model.load_state_dict(torch.load(os.path.join(path_to_folder, model_name), weights_only=True, map_location=torch.device('cpu')))
    nn_model.eval()
    return nn_model

In [56]:
def get_nodes_str(nodes):
    if type(nodes) is list:
        return "_".join([str(node) for node in nodes])
    else:
        return str(nodes)

In [57]:
def adjust_model_to_batch(batch_number, labels, batches):
    batch = batches[batch_number]
    for label in labels:
        value = batch[make_label_from_changing_var(label)].iloc[0]
        basico.set_parameters(make_constant_input_label(label), initial_value=value)

In [58]:
def make_constant_input_label(label):
    return f"{label}0"

In [59]:
true_labels = ["X True", "S true", "PT true", "PM true", "V true"]

In [60]:
def calculate_basico_prediction(document_name, path_to_folder):
    duration = len(Xs_ys[0][1]) - 1
    step_number = duration

    basico.load_model(os.path.join(path_to_folder, document_name))

    copasi_preds = []
    for batch_number in range(len(batches)):
        adjust_model_to_batch(batch_number, changing_vars, batches)
        df = basico.run_time_course(step_number=step_number, duration=duration)
        copasi_pred = torch.tensor(df[concentrations_symbols].values)
        copasi_preds.append(copasi_pred)
    copasi_preds = torch.stack(copasi_preds)
    return copasi_preds

In [61]:
def calculate_ann_predictions(nn_model, X_all):
    y_pred = nn_model(X_all)
    y_pred_scaled = y_pred * (maxes_tensor - mins_tensor) + mins_tensor
    return y_pred_scaled

In [62]:
model_encodings = {"ffnn": ["big_assignment_rules", "layers_as_compartments", "nodes_as_parameters", "nodes_as_species"],
                   "ernn": ["big_event_assignments", "layers_as_compartments", "nodes_as_parameters", "nodes_as_species"]}
def get_sbml_filenames(model_type, activation, layers):
    if model_type == "rnn":
        model_type = "ernn"
    return [f"PR_{model_type}_{activation}_{get_nodes_str(layers)}_{encoding}_sbml.xml" for encoding in model_encodings[model_type]]

In [63]:
# load models
path_to_folder = "../pr_best_models"
path_to_sbml_folder = "../pr_different_encodings"
with torch.no_grad():
    for model_type, nodes_s in num_nodes.items():
        for activation in activations:
            for layers in nodes_s:
                model_name = f"PR_{model_type}_{activation}_wmse_{get_nodes_str(layers)}_best"
                nn_model = load_model(model_name, path_to_folder, model_type, activation, layers)
                pred = calculate_ann_predictions(nn_model, X_all)
                sbml_names = get_sbml_filenames(model_type, activation, layers)
                for sbml_name in sbml_names:
                    copasi_preds = calculate_basico_prediction(sbml_name, path_to_sbml_folder)
                    mse = torch.nn.functional.mse_loss(pred, copasi_preds)
                    print(sbml_name, mse.item())

PR_ffnn_tanh_13_13_big_assignment_rules_sbml.xml 4.144048178821836e-13
PR_ffnn_tanh_13_13_layers_as_compartments_sbml.xml 4.144048178821836e-13
PR_ffnn_tanh_13_13_nodes_as_parameters_sbml.xml 4.144048178821836e-13
PR_ffnn_tanh_13_13_nodes_as_species_sbml.xml 4.144048178821836e-13
PR_ffnn_tanh_15_15_big_assignment_rules_sbml.xml 2.6919954976768387e-13
PR_ffnn_tanh_15_15_layers_as_compartments_sbml.xml 2.6919954976768387e-13
PR_ffnn_tanh_15_15_nodes_as_parameters_sbml.xml 2.6919954976768387e-13
PR_ffnn_tanh_15_15_nodes_as_species_sbml.xml 2.6919954976768387e-13
PR_ffnn_tanh_13_13_13_big_assignment_rules_sbml.xml 3.70951335685052e-13
PR_ffnn_tanh_13_13_13_layers_as_compartments_sbml.xml 3.70951335685052e-13
PR_ffnn_tanh_13_13_13_nodes_as_parameters_sbml.xml 3.70951335685052e-13
PR_ffnn_tanh_13_13_13_nodes_as_species_sbml.xml 3.70951335685052e-13
PR_ffnn_relu_13_13_big_assignment_rules_sbml.xml 2.5347249405878263e-13
PR_ffnn_relu_13_13_layers_as_compartments_sbml.xml 2.5347249405878263e-13